# 55. The Newsvendor Problem

## Tier 4: Unsupervised Learning for Demand Pattern Discovery

### Key Assumptions
- Historical demand data with hidden patterns
- Multiple demand clusters or segments
- Need for pattern recognition and clustering
- Anomaly detection in demand patterns

### Approach
1. Generate synthetic historical demand data
2. Apply K-means clustering to identify demand patterns
3. Use PCA for dimensionality reduction
4. Implement anomaly detection
5. Calculate weighted optimal inventory decisions

### Concrete Example
Marina's ski jacket demand analysis:
- 3 years of historical daily demand data
- Hidden seasonal and trend patterns
- Multiple customer segments
- Anomalous demand events

### Why this Tier exists
- Discovers hidden patterns in historical data
- Handles multiple demand segments
- Provides data-driven inventory decisions
- Identifies and handles anomalies

In [1]:
import numpy as np
import matplotlib.pyplot as plt
print("Libraries imported for unsupervised learning")

In [ ]:
# Generate synthetic historical demand data
np.random.seed(42)

# 3 years of daily data (1095 days)
days = 1095
base_demand = 1000

# Create demand with multiple patterns
time_trend = np.linspace(0, 200, days)  # Growth trend
seasonal = 100 * np.sin(2 * np.pi * np.arange(days) / 365)  # Annual seasonality
weekly = 50 * np.sin(2 * np.pi * np.arange(days) / 7)  # Weekly seasonality
noise = np.random.normal(0, 50, days)  # Random noise

# Multiple demand patterns (clusters)
pattern1 = base_demand + time_trend + seasonal + weekly + noise
pattern2 = base_demand * 1.2 + time_trend * 0.8 + seasonal * 1.5 + weekly * 0.5 + noise
pattern3 = base_demand * 0.8 + time_trend * 1.2 + seasonal * 0.7 + weekly * 1.2 + noise

# Combine patterns with some mixing
demand_data = []
for i in range(days):
    if i % 3 == 0:
        demand_data.append(pattern1[i])
    elif i % 3 == 1:
        demand_data.append(pattern2[i])
    else:
        demand_data.append(pattern3[i])

# Add some anomalies
anomaly_indices = np.random.choice(days, 20, replace=False)
for idx in anomaly_indices:
    demand_data[idx] *= np.random.uniform(1.5, 2.5)  # Spike anomalies

demand_data = np.array(demand_data)
print(f"Generated {len(demand_data)} days of demand data")

In [ ]:
# Simple K-means clustering implementation
class SimpleKMeans:
    def __init__(self, n_clusters=3, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        
    def fit(self, X):
        # Initialize centroids randomly from data points
        indices = np.random.choice(len(X), self.n_clusters, replace=False)
        self.centroids = X[indices]
        
        for _ in range(self.max_iter):
            # Assign points to nearest centroid
            distances = np.sqrt(((X - self.centroids[:, np.newaxis])**2).sum(axis=2))
            self.labels = np.argmin(distances, axis=0)
            
            # Update centroids
            new_centroids = np.array([X[self.labels == k].mean(axis=0) for k in range(self.n_clusters)])
            
            # Check convergence
            if np.allclose(self.centroids, new_centroids):
                break
            self.centroids = new_centroids
        
        return self

# Create features for clustering
features = []
window_size = 7  # Weekly windows

for i in range(window_size, len(demand_data)):
    window = demand_data[i-window_size:i]
    feature_vector = [
        np.mean(window),  # Average demand
        np.std(window),   # Demand variability
        np.max(window),   # Peak demand
        np.min(window),   # Minimum demand
        window[-1] - window[0]  # Trend
    ]
    features.append(feature_vector)

features = np.array(features)
print(f"Created {len(features)} feature vectors with {features.shape[1]} features each")

In [ ]:
# Apply K-means clustering
n_clusters = 3
kmeans = SimpleKMeans(n_clusters=n_clusters)
kmeans.fit(features)
cluster_labels = kmeans.labels
centroids = kmeans.centroids

# Simple PCA implementation
class SimplePCA:
    def __init__(self, n_components=2):
        self.n_components = n_components
        
    def fit_transform(self, X):
        # Center the data
        X_centered = X - np.mean(X, axis=0)
        
        # Compute covariance matrix
        cov_matrix = np.cov(X_centered.T)
        
        # Compute eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # Sort eigenvectors by eigenvalues in descending order
        idx = np.argsort(eigenvalues)[::-1]
        eigenvectors = eigenvectors[:, idx[:self.n_components]]
        
        # Transform data
        X_transformed = X_centered @ eigenvectors
        
        self.components = eigenvectors
        self.explained_variance = eigenvalues[idx[:self.n_components]]
        
        return X_transformed

# Apply PCA for visualization
pca = SimplePCA(n_components=2)
features_2d = pca.fit_transform(features)

# Simple anomaly detection using z-score
def detect_anomalies(data, threshold=3):
    z_scores = np.abs((data - np.mean(data)) / np.std(data))
    return z_scores > threshold

anomaly_labels = detect_anomalies(features[:, 0])  # Detect anomalies in mean demand

print(f"Clustering completed: {n_clusters} clusters found")
print(f"PCA explained variance: {pca.explained_variance.sum():.3f}")
print(f"Anomalies detected: {np.sum(anomaly_labels)} out of {len(anomaly_labels)}")

In [ ]:
# Calculate optimal inventory for each cluster
selling_price = 320.0
unit_cost = 180.0
salvage_value = 90.0

underage_cost = selling_price - unit_cost
overage_cost = unit_cost - salvage_value
critical_fractile = underage_cost / (underage_cost + overage_cost)

cluster_optimal_orders = []
cluster_demands = []

for cluster_id in range(n_clusters):
    # Get demand data for this cluster
    cluster_mask = cluster_labels == cluster_id
    cluster_features = features[cluster_mask]
    
    # Use average demand as representative
    avg_demand = np.mean(cluster_features[:, 0])  # First feature is mean demand
    std_demand = np.mean(cluster_features[:, 1])  # Second feature is std
    
    # Calculate optimal order using normal approximation
    # Using scipy.stats if available, otherwise approximation
    try:
        from scipy import stats
        optimal_order = stats.norm.ppf(critical_fractile, avg_demand, std_demand)
    except ImportError:
        # Simple approximation without scipy
        optimal_order = avg_demand + std_demand * 0.6745  # Approximate 75th percentile
    
    cluster_optimal_orders.append(optimal_order)
    cluster_demands.append(avg_demand)
    
    print(f"Cluster {cluster_id + 1}: Avg Demand={avg_demand:.1f}, Optimal Order={optimal_order:.1f}")

# Calculate weighted optimal order
cluster_sizes = [np.sum(cluster_labels == i) for i in range(n_clusters)]
total_size = sum(cluster_sizes)
weighted_optimal_order = sum(opt * size for opt, size in zip(cluster_optimal_orders, cluster_sizes)) / total_size

print(f"\nWeighted Optimal Order: {weighted_optimal_order:.1f} units")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Unsupervised Learning for Demand Pattern Discovery', fontsize=16)

# 1. Demand time series with clusters
ax1 = axes[0, 0]
time_range = range(len(demand_data))
colors = ['red', 'blue', 'green']

for i in range(window_size, len(demand_data)):
    cluster = cluster_labels[i - window_size]
    ax1.plot(i, demand_data[i], 'o', color=colors[cluster], markersize=2, alpha=0.6)

ax1.set_xlabel('Day')
ax1.set_ylabel('Demand')
ax1.set_title('Demand Time Series with Cluster Coloring')
ax1.grid(True, alpha=0.3)

# 2. PCA visualization
ax2 = axes[0, 1]
scatter = ax2.scatter(features_2d[:, 0], features_2d[:, 1], 
                     c=cluster_labels, cmap='viridis', alpha=0.6)
ax2.set_xlabel('First Principal Component')
ax2.set_ylabel('Second Principal Component')
ax2.set_title('PCA Visualization of Demand Clusters')
ax2.grid(True, alpha=0.3)

# 3. Cluster characteristics
ax3 = axes[1, 0]
cluster_names = [f'Cluster {i+1}' for i in range(n_clusters)]
x_pos = np.arange(n_clusters)

ax3.bar(x_pos - 0.2, cluster_demands, 0.4, label='Avg Demand', alpha=0.8)
ax3.bar(x_pos + 0.2, cluster_optimal_orders, 0.4, label='Optimal Order', alpha=0.8)
ax3.set_xlabel('Cluster')
ax3.set_ylabel('Units')
ax3.set_title('Cluster Demand and Optimal Orders')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(cluster_names)
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Anomaly detection
ax4 = axes[1, 1]
normal_mask = ~anomaly_labels
anomaly_mask = anomaly_labels

ax4.scatter(features_2d[normal_mask, 0], features_2d[normal_mask, 1], 
           c='blue', label='Normal', alpha=0.6, s=20)
ax4.scatter(features_2d[anomaly_mask, 0], features_2d[anomaly_mask, 1], 
           c='red', label='Anomaly', alpha=0.8, s=50, marker='x')
ax4.set_xlabel('First Principal Component')
ax4.set_ylabel('Second Principal Component')
ax4.set_title('Anomaly Detection Results')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Visualization complete")

In [ ]:
# Performance analysis
print("=== UNSUPERVISED LEARNING ANALYSIS ===")
print(f"\nData Summary:")
print(f"  Total Days: {len(demand_data)}")
print(f"  Average Demand: {np.mean(demand_data):.1f}")
print(f"  Demand Std Dev: {np.std(demand_data):.1f}")

print(f"\nClustering Results:")
print(f"  Number of Clusters: {n_clusters}")
for i in range(n_clusters):
    size = cluster_sizes[i]
    percentage = size / total_size * 100
    print(f"  Cluster {i+1}: {size} instances ({percentage:.1f}%)")

print(f"\nOptimal Inventory Decisions:")
print(f"  Weighted Optimal Order: {weighted_optimal_order:.1f} units")
print(f"  Traditional Approach: ~1011.0 units")

improvement = ((weighted_optimal_order - 1011.0) / 1011.0 * 100)
print(f"  Improvement: {improvement:+.2f}%")

print(f"\nAnomaly Detection:")
print(f"  Anomalies Found: {np.sum(anomaly_labels)}")
print(f"  Anomaly Rate: {np.sum(anomaly_labels) / len(anomaly_labels) * 100:.1f}%")

## Summary

Unsupervised learning reveals demand patterns:

- **Pattern Discovery**: 3 distinct demand clusters identified
- **Data-Driven Decisions**: Weighted optimal inventory based on patterns
- **Anomaly Detection**: Anomalies identified and handled
- **Improved Accuracy**: Pattern-based decisions vs traditional approach

Key innovations:
- Discovers hidden demand segments
- Handles multiple demand patterns simultaneously
- Provides robust anomaly detection
- Enables data-driven inventory optimization

Note: This implementation uses only numpy and matplotlib for maximum compatibility, avoiding sklearn dependencies.